# SB100 Squad 2: vetorizar um pacote baixado do site

O caminho é:

1. No site, aba **Vetorização**, escolha a coleção, marque os documentos e baixe
   o pacote (`pacote_....zip`). Ele leva o texto extraído (com as correções
   feitas à mão), os metadados da curadoria, a coleção e o endereço do Qdrant.
2. Aqui, envie esse zip, informe a chave do Qdrant e rode as células. Os
   vetores vão direto para a coleção.
3. No site, aba **Vetorização**, clique em **Atualizar** e **Confirmar**: o site
   confere os pontos no Qdrant e registra quais documentos foram vetorizados.

A chave do Qdrant não fica no notebook nem no pacote. Guarde-a nos Secrets do
Colab (ícone de chave na barra lateral) com o nome `QDRANT_API_KEY`, ou digite
quando a célula 3 pedir.

Sem acesso ao Qdrant daqui? Ponha `ENVIAR_DIRETO = False`: sai um
`vetores_....zip` para enviar na aba Vetorização, e o site grava.

Fatiamento: 1024 caracteres com 20% de sobreposição; tabela e figura entram
inteiras, cada uma num trecho. Modelos: `Qwen/Qwen3-Embedding-0.6B` (denso,
1024 dimensões) e `opensearch-neural-sparse-encoding-doc-v2-distill` (esparso),
os mesmos das coleções que já existem.

Roda no Colab (escolha **GPU** em Ambiente de execução, Alterar tipo) ou num
Jupyter de servidor com placa. Se a sessão cair, rode de novo com o mesmo
pacote: o documento que já terminou é pulado.

## 1. Preparo

In [ ]:
import os, sys, importlib.util
DENTRO_DO_COLAB = "google.colab" in sys.modules
BASE = "/content" if DENTRO_DO_COLAB else os.path.abspath("trabalho")
os.makedirs(BASE, exist_ok=True)
os.environ["SB100_DIR"] = BASE

# o Colab já traz torch e transformers; só instala o que faltar
if importlib.util.find_spec("sentence_transformers") is None:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers>=3.0"], check=True)

import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "SEM GPU (no Colab, troque em Ambiente de execucao); na CPU fica muito lento")

%cd {BASE}
!rm -rf {BASE}/repo
!git clone -q https://github.com/nicolasaws1/parser_rag_framework.git {BASE}/repo
print("pasta de trabalho:", BASE)

## 2. Enviar o pacote

No Colab abre um seletor: escolha o `pacote_....zip` baixado do site.
Fora do Colab, preencha `CAMINHO_DO_PACOTE`.

In [ ]:
CAMINHO_DO_PACOTE = ""  #@param {type:"string"}

if DENTRO_DO_COLAB and not CAMINHO_DO_PACOTE:
    from google.colab import files
    enviados = files.upload()
    CAMINHO_DO_PACOTE = os.path.join(BASE, next(iter(enviados)))
assert CAMINHO_DO_PACOTE.lower().endswith(".zip"), "envie o pacote_....zip baixado do site"
assert os.path.exists(CAMINHO_DO_PACOTE), f"nao achei {CAMINHO_DO_PACOTE}"

import json, zipfile
with zipfile.ZipFile(CAMINHO_DO_PACOTE) as z:
    man = json.loads(z.read("pacote.json"))
print("pacote:", man["pacote"])
print("colecao de destino:", man["colecao"])
print("documentos:", len(man["documentos"]))

## 3. Destino e chave

**COLECAO**: vazio usa a coleção escolhida no site quando o pacote foi baixado.
Preencha para mandar para outra (ela precisa existir; crie pela aba Vetorização).

**SUBSTITUIR**: documento que já tem pontos na coleção é pulado. Marque para
apagar os pontos dele e gravar os novos.

In [ ]:
ENVIAR_DIRETO = True  #@param {type:"boolean"}
COLECAO = ""          #@param {type:"string"}
SUBSTITUIR = False    #@param {type:"boolean"}

if ENVIAR_DIRETO and not os.environ.get("QDRANT_API_KEY"):
    chave = ""
    if DENTRO_DO_COLAB:
        try:
            from google.colab import userdata
            chave = userdata.get("QDRANT_API_KEY") or ""
        except Exception:
            chave = ""
    if not chave:
        from getpass import getpass
        chave = getpass("chave do Qdrant (QDRANT_API_KEY): ").strip()
    os.environ["QDRANT_API_KEY"] = chave
    del chave

flags = ""
if ENVIAR_DIRETO:
    flags += " --enviar"
    if SUBSTITUIR:
        flags += " --substituir"
if COLECAO.strip():
    flags += " --colecao " + COLECAO.strip()
print("destino:", COLECAO.strip() or man["colecao"], "| direto no Qdrant:", ENVIAR_DIRETO)

## 4. Fatiar e vetorizar

**LOTE**: trechos por chamada ao modelo. 32 cabe numa T4; numa L4 ou A100 dá
para subir para 64 ou 128. Se estourar memória, desça.

Na primeira vez baixa os dois modelos (cerca de 1,5 GB).

In [ ]:
LOTE = 32  #@param {type:"integer"}

!python {BASE}/repo/vetorizador/rodar_vetorizacao.py {CAMINHO_DO_PACOTE} --lote {LOTE} {flags}

## 5. Baixar o resultado (só sem envio direto)

Com `ENVIAR_DIRETO = True` não há o que baixar: vá à aba **Vetorização** e
confirme. Sem envio direto, este é o zip para enviar lá.

In [ ]:
import glob
feitos = sorted(glob.glob(os.path.join(BASE, "vetores_*.zip")), key=os.path.getmtime)
assert feitos, "nenhum vetores_*.zip: veja a saida da celula 4"
RESULTADO = feitos[-1]
print("resultado:", RESULTADO, f"({os.path.getsize(RESULTADO)/2**20:.1f} MB)")
if ENVIAR_DIRETO:
    print("ja esta no Qdrant: confirme na aba Vetorizacao do site (o zip fica so como copia)")
elif DENTRO_DO_COLAB:
    from google.colab import files
    files.download(RESULTADO)

## 6. Liberar a GPU (opcional)

No Colab, encerra a sessão para não consumir unidades de computação à toa.

In [ ]:
if DENTRO_DO_COLAB:
    from google.colab import runtime
    runtime.unassign()